# Sampling: temperature, top-p, and determinism

**Session 1 · Foundations · small model (`llama3.2:3b`)**

Feel how the decoding knobs change outputs — and see, from counts, that **temperature 0 is
not the same as deterministic**. On a short factual prompt it effectively is. On an
open-ended one it is not, because temperature is only one of several sampling knobs and a
long generation amplifies tiny differences. This is *why* the eval harness runs things more
than once.

In [1]:
import sys; sys.path.append('..')  # so `utils` and `eval` import from the repo root
from utils import ask, SMALL_MODEL


Run the SAME prompt several times at two temperatures and eyeball the variability.

In [2]:
prompt = "Give me a one-line startup idea for a bakery."
print("--- temperature 0 ---")
for _ in range(3): print("  ", ask(prompt, temperature=0, model=SMALL_MODEL)[:100])
print("--- temperature 1 ---")
for _ in range(3): print("  ", ask(prompt, temperature=1, model=SMALL_MODEL)[:100])

--- temperature 0 ---


   "Rise & Shine" is a subscription-based bakery service that delivers artisanal, small-batch pastries 


   "Rise & Shine" is a subscription-based bakery service that delivers artisanal, small-batch pastries 


   "Rise & Shine" is a subscription-based bakery service that delivers artisanal, small-batch pastries 
--- temperature 1 ---


   Your startup idea for a bakery is to specialize in artisanal "memory pastries" - uniquely crafted de


   Here's a one-line startup idea for a bakery: "SweetStacks" is a subscription-based bakery that deliv


   Here's a one-line startup idea for a bakery:

"Sweet Escape" - a mobile bakery that offers a unique,


### Worked example

Count *distinct* answers over N runs, for a **factual** prompt and a **creative** prompt, at
three temperatures. Watch the two prompts behave completely differently at temperature 0.

In [3]:
# Worked example: distinct-answer count at each temperature
def distinct(prompt, temperature, n=6):
    outs = [ask(prompt, temperature=temperature, model=SMALL_MODEL).strip() for _ in range(n)]
    return len(set(outs)), n

FACTUAL  = "What is 17 * 23? Answer with the number only."
CREATIVE = "Give me a one-line startup idea for a bakery."

for label, p in [("factual ", FACTUAL), ("creative", CREATIVE)]:
    row = "  ".join(f"temp {t}: {d}/{n}" for t in (0.0, 0.7, 1.0) for d, n in [distinct(p, t)])
    print(f"{label}  {row}")


factual   temp 0.0: 1/6  temp 0.7: 1/6  temp 1.0: 1/6


creative  temp 0.0: 1/6  temp 0.7: 6/6  temp 1.0: 6/6


## Your turn - vary the example

1. The factual prompt is ~stable at temp 0; the creative one is not. Make the creative prompt
   *more constrained* ("...in exactly 5 words") — does temp-0 variability drop?
2. At what temperature does the factual prompt start giving wrong answers, not just varied ones?
3. When would you want temperature 0 in production, and when would you raise it? Tie each
   answer to one of the numbers above.


In [4]:
# Your variation here - copy the worked example above and change ONE thing, then re-run
